In [ ]:
%pip install -q chess zstandard pandas pyarrow requests tqdm

# data part: lichess database

In [ ]:
import io
import re
import requests
import zstandard as zstd
import chess
import chess.pgn
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path

MONTH = "2024-06"
URL = f"https://database.lichess.org/standard/lichess_db_standard_rated_{MONTH}.pgn.zst"
N_GAMES = 10000
REQUIRE_EVAL = True
MIN_PLIES = 20
MIN_ELO, MAX_ELO = 600, 2800
OUT_DIR = Path("data")
OUT_DIR.mkdir(exist_ok=True)

In [ ]:
def stream_game_texts(url):
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        reader = zstd.ZstdDecompressor().stream_reader(r.raw, read_across_frames=True)
        text = io.TextIOWrapper(reader, encoding="utf-8", errors="ignore")
        buf = []
        for line in text:
            if line.startswith("[Event ") and buf:
                yield "".join(buf)
                buf = []
            buf.append(line)
        if buf:
            yield "".join(buf)


HEADER_RE = re.compile(r'^\[(\w+) "(.*)"\]$', re.M)


def quick_filter(txt):
    h = dict(HEADER_RE.findall(txt))
    event = h.get("Event", "")
    if not (event.startswith("Rated Blitz") or event.startswith("Rated Rapid")):
        return None
    if "BOT" in (h.get("WhiteTitle", ""), h.get("BlackTitle", "")):
        return None
    if h.get("Termination") not in ("Normal", "Time forfeit"):
        return None
    if h.get("Variant", "Standard") != "Standard":
        return None
    try:
        we, be = int(h["WhiteElo"]), int(h["BlackElo"])
    except (KeyError, ValueError):
        return None
    if not (MIN_ELO <= we <= MAX_ELO and MIN_ELO <= be <= MAX_ELO):
        return None
    if "[%clk" not in txt:
        return None
    if REQUIRE_EVAL and "[%eval" not in txt:
        return None
    return h

In [ ]:
PIECE_VALUES = [(chess.KNIGHT, 3), (chess.BISHOP, 3), (chess.ROOK, 5), (chess.QUEEN, 9)]


def parse_game(txt, h):
    game = chess.pgn.read_game(io.StringIO(txt))
    if game is None or game.errors:
        return None
    base, inc = map(int, h["TimeControl"].split("+"))
    gid = h.get("Site", "").rsplit("/", 1)[-1]
    board = game.board()
    last_clock = {chess.WHITE: float(base), chess.BLACK: float(base)}
    prev_eval = None
    rows = []
    for ply, node in enumerate(game.mainline()):
        move = node.move
        mover = board.turn
        clk = node.clock()
        ev = node.eval()
        ev_cp = ev.white().score(mate_score=10000) if ev is not None else None
        npm = sum(len(board.pieces(pt, c)) * v for pt, v in PIECE_VALUES for c in (chess.WHITE, chess.BLACK))
        rows.append(dict(
            game_id=gid,
            ply=ply,
            move_number=ply // 2 + 1,
            side="white" if mover == chess.WHITE else "black",
            fen=board.fen(),
            move=move.uci(),
            piece=board.piece_at(move.from_square).symbol().upper(),
            is_capture=board.is_capture(move),
            is_check=board.gives_check(move),
            n_legal=board.legal_moves.count(),
            non_pawn_material=npm,
            elo_self=int(h["WhiteElo"] if mover else h["BlackElo"]),
            elo_oppo=int(h["BlackElo"] if mover else h["WhiteElo"]),
            clock_before=last_clock[mover],
            clock_after=clk,
            eval_before_cp=prev_eval,
            eval_after_cp=ev_cp,
            base_time=base,
            increment=inc,
            event=h["Event"],
            eco=h.get("ECO"),
            opening=h.get("Opening"),
            result=h["Result"],
            termination=h["Termination"],
        ))
        if clk is not None:
            last_clock[mover] = clk
        prev_eval = ev_cp
        board.push(move)
    if len(rows) < MIN_PLIES:
        return None
    return rows

In [ ]:
games, rows = [], []
scanned = 0
pbar = tqdm(total=N_GAMES)
for txt in stream_game_texts(URL):
    scanned += 1
    h = quick_filter(txt)
    if h is None:
        continue
    r = parse_game(txt, h)
    if r is None:
        continue
    rows.extend(r)
    games.append(dict(
        game_id=r[0]["game_id"],
        white_elo=int(h["WhiteElo"]),
        black_elo=int(h["BlackElo"]),
        event=h["Event"],
        time_control=h["TimeControl"],
        eco=h.get("ECO"),
        opening=h.get("Opening"),
        result=h["Result"],
        termination=h["Termination"],
        n_plies=len(r),
    ))
    pbar.update(1)
    if len(games) >= N_GAMES:
        break
pbar.close()
print(f"scanned {scanned} | kept games {len(games)} | positions {len(rows)}")

In [ ]:
moves = pd.DataFrame(rows)
games_df = pd.DataFrame(games).drop_duplicates("game_id")
moves = moves[moves["game_id"].isin(games_df["game_id"])]

bad_games = moves.loc[moves["clock_after"].isna(), "game_id"].unique()
moves = moves[~moves["game_id"].isin(bad_games)].copy()
games_df = games_df[~games_df["game_id"].isin(bad_games)].copy()

moves["eval_before_cp"] = pd.to_numeric(moves["eval_before_cp"])
moves["eval_after_cp"] = pd.to_numeric(moves["eval_after_cp"])

moves["time_control_type"] = np.where(moves["event"].str.startswith("Rated Blitz"), "blitz", "rapid")
moves["time_spent"] = (moves["clock_before"] - moves["clock_after"] + moves["increment"]).clip(lower=0)
moves["clock_frac"] = moves["clock_before"] / moves["base_time"]
moves["low_time"] = moves["clock_before"] < 30

sign = np.where(moves["side"] == "white", 1, -1)


def win_chance(cp):
    cp = np.clip(cp, -1000, 1000)
    return 2 / (1 + np.exp(-0.00368208 * cp)) - 1


moves["wc_before"] = win_chance(sign * moves["eval_before_cp"])
moves["wc_after"] = win_chance(sign * moves["eval_after_cp"])
moves["wc_drop"] = moves["wc_before"] - moves["wc_after"]
moves["move_quality"] = pd.cut(
    moves["wc_drop"],
    [-np.inf, 0.1, 0.2, 0.3, np.inf],
    labels=["good", "inaccuracy", "mistake", "blunder"],
    right=False,
).astype(str).replace("nan", pd.NA)

moves["phase"] = np.select(
    [moves["move_number"] <= 12, moves["non_pawn_material"] <= 26],
    ["opening", "endgame"],
    "middlegame",
)

moves["elo_band"] = pd.cut(moves["elo_self"], bins=list(range(600, 3000, 200)), right=False).astype(str)
moves["elo_diff"] = moves["elo_self"] - moves["elo_oppo"]
moves["forced"] = moves["n_legal"] == 1

moves = moves.reset_index(drop=True)
print(moves.shape, games_df.shape)

In [ ]:
moves.to_parquet(OUT_DIR / f"moves_{MONTH}.parquet", index=False)
games_df.to_parquet(OUT_DIR / f"games_{MONTH}.parquet", index=False)

In [ ]:
print(moves["time_control_type"].value_counts(), "\n")
print(moves["phase"].value_counts(), "\n")
print(moves["move_quality"].value_counts(dropna=False), "\n")
print(moves["elo_band"].value_counts().sort_index(), "\n")
print(moves[["clock_before", "time_spent", "wc_drop", "n_legal"]].describe().T, "\n")
print("eval coverage:", moves["eval_after_cp"].notna().mean().round(3))
print("forced moves:", moves["forced"].mean().round(3))

# model part: maia-2

In [ ]:
%pip install -q maia2

In [ ]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm.auto import tqdm
from maia2 import model as maia_model
from maia2.inference import TestDataset, prepare, inference_batch
from maia2.utils import mirror_move

MONTH = "2024-06"
DATA_DIR = Path("data")
BATCH_SIZE = 512
NUM_WORKERS = 0
MAX_POSITIONS = None
SEED = 42

moves = pd.read_parquet(DATA_DIR / f"moves_{MONTH}.parquet")
if MAX_POSITIONS is not None and len(moves) > MAX_POSITIONS:
    gids = moves["game_id"].drop_duplicates().sample(frac=1, random_state=SEED)
    sizes = moves.groupby("game_id").size().loc[gids].cumsum()
    moves = moves[moves["game_id"].isin(sizes[sizes <= MAX_POSITIONS].index)].reset_index(drop=True)
print(moves.shape)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
nets = {tc: maia_model.from_pretrained(type=tc, device=device) for tc in ("blitz", "rapid")}
all_moves_dict, elo_dict, all_moves_dict_reversed = prepare()
print(device, len(all_moves_dict), list(elo_dict.keys()))

In [ ]:
import chess
from maia2.utils import board_to_tensor, map_to_category


class MaiaPositions(torch.utils.data.Dataset):

    def __init__(self, fens, elo_self, elo_oppo):
        self.fens = list(fens)
        self.elo_self = [int(e) for e in elo_self]
        self.elo_oppo = [int(e) for e in elo_oppo]

    def __len__(self):
        return len(self.fens)

    def __getitem__(self, i):
        board = chess.Board(self.fens[i])
        if board.turn == chess.BLACK:
            board = board.mirror()
        legal = torch.zeros(len(all_moves_dict))
        legal[[all_moves_dict[m.uci()] for m in board.legal_moves]] = 1
        return (
            board_to_tensor(board),
            map_to_category(self.elo_self[i], elo_dict),
            map_to_category(self.elo_oppo[i], elo_dict),
            legal,
        )


def run_maia(df, net, batch_size=BATCH_SIZE):
    data = df[["fen", "move", "elo_self", "elo_oppo"]].reset_index(drop=True)
    is_black = data["fen"].str.split(" ").str[1].eq("b").to_numpy()
    true_moves = [mirror_move(m) if b else m for m, b in zip(data["move"], is_black)]
    true_idx = np.array([all_moves_dict.get(m, -1) for m in true_moves])
    if (true_idx < 0).any():
        raise ValueError(f"{(true_idx < 0).sum()} moves not in Maia vocabulary")
    loader = torch.utils.data.DataLoader(
        MaiaPositions(data["fen"], data["elo_self"], data["elo_oppo"]),
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )
    dev = next(net.parameters()).device
    net.eval()
    buf = {key: [] for key in ("topk_idx", "topk_p", "p_true", "rank", "entropy", "n_legal", "value")}
    offset = 0
    with torch.no_grad():
        for boards, e_self, e_oppo, legal in tqdm(loader):
            boards, e_self, e_oppo = boards.to(dev), e_self.to(dev), e_oppo.to(dev)
            legal = legal.to(dev).bool()
            logits, _, value = net(boards, e_self, e_oppo)
            probs = logits.masked_fill(~legal, float("-inf")).softmax(-1)
            n = probs.shape[0]
            t = torch.as_tensor(true_idx[offset:offset + n], device=dev).unsqueeze(1)
            p_true = probs.gather(1, t).squeeze(1)
            topk_p, topk_idx = probs.topk(5, dim=1)
            buf["topk_idx"].append(topk_idx.cpu().numpy())
            buf["topk_p"].append(topk_p.cpu().numpy())
            buf["p_true"].append(p_true.cpu().numpy())
            buf["rank"].append(((probs > p_true.unsqueeze(1)).sum(1) + 1).cpu().numpy())
            buf["entropy"].append((-(probs * torch.log(probs.clamp_min(1e-12))).sum(1)).cpu().numpy())
            buf["n_legal"].append(legal.sum(1).cpu().numpy())
            buf["value"].append((value.reshape(-1) / 2 + 0.5).clamp(0, 1).cpu().numpy())
            offset += n
    r = {key: np.concatenate(v) for key, v in buf.items()}
    rev = all_moves_dict_reversed
    top_moves, top_probs = [], []
    for row_i, row_p, b in zip(r["topk_idx"], r["topk_p"], is_black):
        mv = [mirror_move(rev[i]) if b else rev[i] for i, p in zip(row_i, row_p) if p > 0]
        top_moves.append(mv)
        top_probs.append(row_p[:len(mv)].round(5).tolist())
    n_legal = r["n_legal"]
    out = pd.DataFrame({
        "maia_move": [m[0] for m in top_moves],
        "maia_p_top1": r["topk_p"][:, 0],
        "maia_p_true": r["p_true"],
        "maia_rank_true": r["rank"],
        "maia_entropy": r["entropy"],
        "maia_entropy_norm": np.where(n_legal > 1, r["entropy"] / np.log(np.maximum(n_legal, 2)), 0.0),
        "maia_value_mover": r["value"],
        "maia_top5_moves": top_moves,
        "maia_top5_probs": top_probs,
    }, index=df.index)
    out["maia_correct"] = out["maia_move"].to_numpy() == df["move"].to_numpy()
    out["maia_top3_hit"] = out["maia_rank_true"] <= 3
    out["maia_top5_hit"] = out["maia_rank_true"] <= 5
    return out

In [ ]:
blitz_pool = moves[moves["time_control_type"] == "blitz"]
sample = blitz_pool.sample(min(2000, len(blitz_pool)), random_state=SEED)
official_input = sample.rename(
    columns={"fen": "board", "elo_self": "active_elo", "elo_oppo": "opponent_elo"}
)[["board", "move", "active_elo", "opponent_elo"]].reset_index(drop=True)
official, official_acc = inference_batch(
    official_input,
    nets["blitz"],
    verbose=0,
    batch_size=BATCH_SIZE,
    num_workers=0,
)
ours = run_maia(sample, nets["blitz"])
official_top1 = official["move_probs"].map(lambda d: max(d, key=d.get)).to_numpy()
print("official acc:", official_acc, "| ours acc:", round(ours["maia_correct"].mean(), 4))
print("top-1 agreement:", round((official_top1 == ours["maia_move"].to_numpy()).mean(), 4))

In [ ]:
parts = [
    run_maia(moves[moves["time_control_type"] == tc], net)
    for tc, net in nets.items()
    if (moves["time_control_type"] == tc).any()
]
preds = pd.concat(parts).sort_index()
moves = moves.drop(columns=[c for c in preds.columns if c in moves.columns]).join(preds)
print(moves.shape)

In [ ]:
elo_bins = [-np.inf] + list(range(1100, 2001, 100)) + [np.inf]
elo_labels = list(elo_dict.keys())
moves["maia_elo_cat_self"] = pd.cut(moves["elo_self"], elo_bins, right=False, labels=elo_labels).astype(str)
moves["maia_elo_cat_oppo"] = pd.cut(moves["elo_oppo"], elo_bins, right=False, labels=elo_labels).astype(str)
moves["maia_failed"] = ~moves["maia_correct"]
moves.to_parquet(DATA_DIR / f"moves_maia_{MONTH}.parquet", index=False)

In [ ]:
ev = moves[~moves["forced"]]
print("top-1 acc all:", round(moves["maia_correct"].mean(), 4))
print("top-1 acc non-forced:", round(ev["maia_correct"].mean(), 4))
print("top-3 acc non-forced:", round(ev["maia_top3_hit"].mean(), 4), "\n")
print(ev.groupby("time_control_type")["maia_correct"].agg(["mean", "size"]), "\n")
print(ev.groupby("maia_elo_cat_self")["maia_correct"].agg(["mean", "size"]).reindex(elo_labels))

# 2D correlation matrix plots

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, FastICA

TAB_FEATURES = [
    "elo_self",
    "elo_oppo",
    "elo_diff",
    "clock_before",
    "time_spent",
    "move_number",
    "n_legal",
    "non_pawn_material",
    "wc_before",
    "wc_drop",
]
PHASES = ["opening", "middlegame", "endgame"]
N_FIT_PHASE = 30000

pool = moves[~moves["forced"]]
base_all = (~pool["maia_correct"]).mean()
fig, axes = plt.subplots(len(PHASES), 2, figsize=(12, 15))
loadings, explained = {}, {}
for row, ph_name in enumerate(PHASES):
    d = pool.loc[pool["phase"] == ph_name, TAB_FEATURES + ["maia_correct"]].dropna()
    d = d.sample(min(N_FIT_PHASE, len(d)), random_state=42)
    d["time_spent"] = d["time_spent"].clip(upper=d["time_spent"].quantile(0.99))
    Xp = StandardScaler().fit_transform(d[TAB_FEATURES])
    fp = (~d["maia_correct"]).to_numpy().astype(float)
    p = PCA(n_components=2, random_state=42).fit(Xp)
    i = FastICA(n_components=2, random_state=42, whiten="unit-variance", max_iter=1000).fit(Xp)
    for col, (name, Z) in enumerate([("PCA", p.transform(Xp)), ("ICA", i.transform(Xp))]):
        ax = axes[row, col]
        hb = ax.hexbin(Z[:, 0], Z[:, 1], C=fp, reduce_C_function=np.mean, gridsize=40, mincnt=20,
                       cmap="RdBu_r", vmin=base_all - 0.25, vmax=base_all + 0.25)
        fig.colorbar(hb, ax=ax, label="share failed")
        ax.set_title(f"{ph_name} | {name} | failure rate {fp.mean():.2f} | n={len(d)}")
        ax.set_xlabel(f"{name} 1")
        ax.set_ylabel(f"{name} 2")
    loadings[ph_name] = pd.DataFrame(np.hstack([p.components_.T, i.components_.T]),
                                     index=TAB_FEATURES, columns=["PC1", "PC2", "IC1", "IC2"])
    explained[ph_name] = p.explained_variance_ratio_
plt.tight_layout()
plt.show()

for ph_name in PHASES:
    print(f"\n{ph_name} | PCA explained variance: {explained[ph_name].round(3)}")
    print(loadings[ph_name].round(2))

In [ ]:
import chess


def board_vec(fen):
    b = chess.Board(fen)
    if b.turn == chess.BLACK:
        b = b.mirror()
    v = np.zeros(768, dtype=np.float32)
    for sq, pc in b.piece_map().items():
        v[(pc.piece_type - 1 + (0 if pc.color else 6)) * 64 + sq] = 1
    return v


N_BOARD_PHASE = 15000
fig, axes = plt.subplots(1, len(PHASES), figsize=(18, 5.5))
board_explained = {}
for ax, ph_name in zip(axes, PHASES):
    d = pool.loc[pool["phase"] == ph_name, ["fen", "maia_correct"]]
    d = d.sample(min(N_BOARD_PHASE, len(d)), random_state=42)
    Bp = np.stack([board_vec(f) for f in tqdm(d["fen"], desc=ph_name)])
    bp = PCA(n_components=10, random_state=42).fit(Bp)
    Z = bp.transform(Bp)[:, :2]
    fp = (~d["maia_correct"]).to_numpy().astype(float)
    hb = ax.hexbin(Z[:, 0], Z[:, 1], C=fp, reduce_C_function=np.mean, gridsize=40, mincnt=15,
                   cmap="RdBu_r", vmin=base_all - 0.25, vmax=base_all + 0.25)
    fig.colorbar(hb, ax=ax, label="share failed")
    ax.set_title(f"Board PCA | {ph_name} | failure rate {fp.mean():.2f}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    board_explained[ph_name] = bp.explained_variance_ratio_
plt.tight_layout()
plt.show()

for ph_name, ev_ratio in board_explained.items():
    print(f"{ph_name} | board PCA explained variance (first 10): {ev_ratio.round(3)}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

FEATURES = [
    "elo_self",
    "elo_oppo",
    "elo_diff",
    "clock_frac",
    "time_spent",
    "move_number",
    "n_legal",
    "wc_before",
    "wc_drop",
    "maia_entropy_norm",
]
N_SAMPLE = 6000

opening = moves.loc[(~moves["forced"]) & (moves["phase"] == "opening"), FEATURES + ["maia_correct"]].dropna()
plot_df = opening.sample(min(N_SAMPLE, len(opening)), random_state=42)
plot_df["time_spent"] = plot_df["time_spent"].clip(upper=plot_df["time_spent"].quantile(0.99))
plot_df["prediction"] = plot_df["maia_correct"].map({True: "correct", False: "failed"})

g = sns.pairplot(
    plot_df.drop(columns="maia_correct"),
    hue="prediction",
    hue_order=["correct", "failed"],
    palette={"correct": "#2a78d6", "failed": "#eb6834"},
    corner=True,
    height=1.6,
    plot_kws=dict(s=6, alpha=0.35, linewidth=0),
    diag_kind="kde",
    diag_kws=dict(common_norm=False, fill=True, alpha=0.3),
)
g.fig.suptitle(
    f"Opening positions: pairwise features coloured by Maia-2 prediction "
    f"(failure rate {(~opening['maia_correct']).mean():.2f}, n={len(opening)})",
    y=1.01,
)
plt.show()

In [ ]:
import shutil
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import chess
import chess.svg
import matplotlib as mpl
from IPython.display import HTML, display
from sklearn.decomposition import PCA
from tqdm.auto import tqdm

REPORT_NAME = "maia2_phase_report"
REPORT_TITLE = "Maia-2 failures by game phase"
GROUP_LABEL = "Phase"
N_BOARD = 20000
N_EXAMPLES = 8
SHOW = {"pca": True, "occupancy": True, "squares": True, "examples": True}

base = moves[~moves["forced"]].copy()
base["time_pressure"] = pd.cut(base["clock_before"], [-np.inf, 30, 60, np.inf], labels=["<30s", "30-60s", ">60s"]).astype(str)
GROUPS = {ph: base[base["phase"] == ph] for ph in ["opening", "middlegame", "endgame"]}
MIX_COL = "time_pressure"

PIECE_LABELS = ["pawn", "knight", "bishop", "rook", "queen", "king"]
BOARD_COLORS = {"square light": "#f4f3ef", "square dark": "#dddbd3", "margin": "#ffffff", "coord": "#52514e"}
BLUE_A, ORANGE_A = "#2a78d6dd", "#eb6834dd"
HEAT_SIZE, EXAMPLE_SIZE = 150, 230
REPORT = []


def emit(html):
    REPORT.append(html)
    display(HTML(html))


def log(text):
    print(text)
    REPORT.append(f'<div class="block" style="font:12px monospace;color:#52514e;margin:4px 0">{text}</div>')


def board_vec(fen):
    b = chess.Board(fen)
    if b.turn == chess.BLACK:
        b = b.mirror()
    v = np.zeros(768, dtype=np.float32)
    for sq, pc in b.piece_map().items():
        v[(pc.piece_type - 1 + (0 if pc.color else 6)) * 64 + sq] = 1
    return v


def heat_svg(values, vmin, vmax, cmap="RdBu_r", size=HEAT_SIZE, alpha=0.9):
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    cm = mpl.colormaps[cmap]
    fill = {sq: mpl.colors.to_hex(cm(norm(values[sq]))) + f"{int(alpha * 255):02x}"
            for sq in range(64) if np.isfinite(values[sq])}
    return chess.svg.board(chess.Board(None), fill=fill, size=size, coordinates=True, colors=BOARD_COLORS)


def colorbar_html(vmin, vmax, cmap="RdBu_r", label=""):
    stops = ",".join(mpl.colors.to_hex(mpl.colormaps[cmap](t)) for t in np.linspace(0, 1, 11))
    return (f'<div style="font:12px sans-serif;margin:6px 0;color:#52514e">{label}'
            f'<div style="display:flex;align-items:center;gap:6px;margin-top:4px"><span>{vmin:.3g}</span>'
            f'<div style="width:240px;height:10px;background:linear-gradient(90deg,{stops})"></div>'
            f'<span>{vmax:.3g}</span></div></div>')


def show_grid(items, cols=6, title=None, note=None, cap_width=HEAT_SIZE + 20):
    cells = "".join(
        f'<div style="text-align:center;font:11px sans-serif">{svg}'
        f'<div style="max-width:{cap_width}px;margin:auto">{cap}</div></div>'
        for svg, cap in items
    )
    head = f'<div style="font:bold 14px sans-serif;margin:12px 0 6px">{title}</div>' if title else ""
    emit(f'<div class="block">{head}<div style="display:grid;grid-template-columns:repeat({cols},auto);'
         f'gap:6px;justify-content:start">{cells}</div>{note or ""}</div>')


def piece_grid(mat, lim, cmap="RdBu_r"):
    return [(heat_svg(mat[offset + j], -lim, lim, cmap), f"{side} {lab}")
            for side, offset in (("own", 0), ("opponent", 6))
            for j, lab in enumerate(PIECE_LABELS)]


def board_report(name, pool, first=False):
    cls = "" if first else ' class="section"'
    emit(f'<h2{cls} style="font:bold 18px sans-serif;margin:24px 0 4px;border-bottom:1px solid #dddbd3">'
         f'{GROUP_LABEL}: {name}</h2>')
    fails = pool[~pool["maia_correct"]]
    mix = f" | {MIX_COL} mix {pool[MIX_COL].value_counts(normalize=True).round(2).to_dict()}" if MIX_COL else ""
    log(f"{name}: {len(pool)} non-forced positions | failure rate {(~pool['maia_correct']).mean():.3f}{mix}")

    if SHOW["pca"] or SHOW["occupancy"]:
        sub = pool.sample(min(N_BOARD, len(pool)), random_state=42)
        B = np.stack([board_vec(f) for f in tqdm(sub["fen"], desc=f"encoding {name}")])
        failed = ~sub["maia_correct"].to_numpy()

    if SHOW["pca"]:
        bpca = PCA(n_components=5, random_state=42).fit(B)
        scores = bpca.transform(B)
        for k in range(2):
            comp = bpca.components_[k].reshape(12, 64)
            lim = np.abs(comp).max()
            show_grid(
                piece_grid(comp, lim),
                title=f"PC{k + 1} loadings, {name} ({bpca.explained_variance_ratio_[k]:.1%} of variance), side to move at the bottom",
                note=colorbar_html(-lim, lim, label="red = a piece on this square raises the PC score, blue = lowers it"),
            )
            dec = pd.Series(failed).groupby(pd.qcut(scores[:, k], 10, labels=False, duplicates="drop")).mean()
            log(f"{name} PC{k + 1} failure rate by score decile (low -> high): {dec.round(3).tolist()}")

    if SHOW["occupancy"]:
        diff = (B[failed].mean(0) - B[~failed].mean(0)).reshape(12, 64) * 100
        lim = np.abs(diff).max()
        show_grid(
            piece_grid(diff, lim),
            title=f"Where pieces stand when Maia-2 fails vs. succeeds ({name}, n={len(sub)})",
            note=colorbar_html(-lim, lim, label="percentage points: red = occupied more often in failed positions, blue = in correct ones"),
        )

    if SHOW["squares"]:
        fails_black = fails["side"].eq("black").to_numpy()

        def square_share(ucis, part):
            c = np.zeros(64)
            for u, is_black in zip(ucis, fails_black):
                m = chess.Move.from_uci(u)
                sq = m.from_square if part == "from" else m.to_square
                c[chess.square_mirror(sq) if is_black else sq] += 1
            return c / c.sum() * 100

        for part in ("from", "to"):
            human = square_share(fails["move"], part)
            maia = square_share(fails["maia_move"], part)
            top = max(human.max(), maia.max())
            d = human - maia
            dl = np.abs(d).max()
            show_grid(
                [
                    (heat_svg(human, 0, top, "Blues"), f"human '{part}' squares"),
                    (heat_svg(maia, 0, top, "Blues"), f"Maia-2 '{part}' squares"),
                    (heat_svg(d, -dl, dl, "RdBu_r"), "human minus Maia-2"),
                ],
                cols=3,
                title=f"Failed predictions, {name}: '{part}' squares (n={len(fails)}, mover at the bottom)",
                note=colorbar_html(0, top, "Blues", "share of moves in %")
                + colorbar_html(-dl, dl, label="red = humans use this square more, blue = Maia-2 expects it more (pp)"),
            )

    if SHOW["examples"]:
        ex = fails.drop_duplicates("fen").nlargest(N_EXAMPLES, "maia_p_top1")
        items = []
        for _, r in ex.iterrows():
            b = chess.Board(r["fen"])
            hm, mm = chess.Move.from_uci(r["move"]), chess.Move.from_uci(r["maia_move"])
            svg = chess.svg.board(
                b,
                arrows=[chess.svg.Arrow(mm.from_square, mm.to_square, color=BLUE_A),
                        chess.svg.Arrow(hm.from_square, hm.to_square, color=ORANGE_A)],
                size=EXAMPLE_SIZE,
                coordinates=True,
                flipped=b.turn == chess.BLACK,
            )
            cap = (f"{r['side']} to move, {r['elo_self']} vs {r['elo_oppo']}, {r['clock_before']:.0f}s left<br>"
                   f"Maia-2 {r['maia_move']} ({r['maia_p_top1']:.0%}) | human {r['move']} ({r['maia_p_true']:.0%})<br>"
                   f"{r['opening']}")
            items.append((svg, cap))
        show_grid(
            items,
            cols=4,
            cap_width=EXAMPLE_SIZE,
            title=f"Most confident Maia-2 failures ({name})",
            note='<div style="font:12px sans-serif;color:#52514e;margin-top:6px">blue arrow = Maia-2 top prediction, orange arrow = move actually played</div>',
        )


def find_browser():
    candidates = ["/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"]
    found = next((c for c in candidates if Path(c).exists()), None)
    return found or next((shutil.which(n) for n in ("google-chrome", "chromium", "chromium-browser", "chrome") if shutil.which(n)), None)


def export_report(stem, title):
    css = ("@page{size:A4 landscape;margin:10mm}"
           "body{font-family:sans-serif;color:#0b0b0b;background:#ffffff}"
           ".section{break-before:page}"
           ".block{break-inside:avoid;page-break-inside:avoid;margin-bottom:8px}")
    html = (f'<!doctype html><html><head><meta charset="utf-8"><title>{title}</title><style>{css}</style></head>'
            f'<body><h1 style="font:bold 22px sans-serif">{title}</h1>{"".join(REPORT)}</body></html>')
    html_path = Path(f"{stem}.html").resolve()
    pdf_path = Path(f"{stem}.pdf").resolve()
    html_path.write_text(html, encoding="utf-8")
    browser = find_browser()
    if browser:
        subprocess.run(
            [browser, "--headless=new", "--disable-gpu", "--no-pdf-header-footer",
             f"--print-to-pdf={pdf_path}", html_path.as_uri()],
            capture_output=True,
            timeout=300,
        )

for i, (name, pool) in enumerate(GROUPS.items()):
    board_report(name, pool, first=i == 0)
export_report(REPORT_NAME, REPORT_TITLE)

In [ ]:
import shutil
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import chess
import chess.svg
import matplotlib as mpl
from IPython.display import HTML, display
from sklearn.decomposition import PCA
from tqdm.auto import tqdm

REPORT_NAME = "maia2_time_pressure_report"
REPORT_TITLE = "Maia-2 failures by time pressure"
N_BOARD = 20000
N_EXAMPLES = 8
SHOW = {"pca": True, "occupancy": True, "squares": True, "examples": True}

base = moves[~moves["forced"]]
GROUPS = {
    "more than 1 min": base[base["clock_before"] > 60],
    "30 s to 1 min": base[(base["clock_before"] <= 60) & (base["clock_before"] > 30)],
    "less than 30 s": base[base["clock_before"] <= 30],
}

PIECE_LABELS = ["pawn", "knight", "bishop", "rook", "queen", "king"]
BOARD_COLORS = {"square light": "#f4f3ef", "square dark": "#dddbd3", "margin": "#ffffff", "coord": "#52514e"}
BLUE_A, ORANGE_A = "#2a78d6dd", "#eb6834dd"
HEAT_SIZE, EXAMPLE_SIZE = 150, 230
REPORT = []


def emit(html):
    REPORT.append(html)
    display(HTML(html))


def log(text):
    print(text)
    REPORT.append(f'<div class="block" style="font:12px monospace;color:#52514e;margin:4px 0">{text}</div>')


def board_vec(fen):
    b = chess.Board(fen)
    if b.turn == chess.BLACK:
        b = b.mirror()
    v = np.zeros(768, dtype=np.float32)
    for sq, pc in b.piece_map().items():
        v[(pc.piece_type - 1 + (0 if pc.color else 6)) * 64 + sq] = 1
    return v


def heat_svg(values, vmin, vmax, cmap="RdBu_r", size=HEAT_SIZE, alpha=0.9):
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    cm = mpl.colormaps[cmap]
    fill = {sq: mpl.colors.to_hex(cm(norm(values[sq]))) + f"{int(alpha * 255):02x}"
            for sq in range(64) if np.isfinite(values[sq])}
    return chess.svg.board(chess.Board(None), fill=fill, size=size, coordinates=True, colors=BOARD_COLORS)


def colorbar_html(vmin, vmax, cmap="RdBu_r", label=""):
    stops = ",".join(mpl.colors.to_hex(mpl.colormaps[cmap](t)) for t in np.linspace(0, 1, 11))
    return (f'<div style="font:12px sans-serif;margin:6px 0;color:#52514e">{label}'
            f'<div style="display:flex;align-items:center;gap:6px;margin-top:4px"><span>{vmin:.3g}</span>'
            f'<div style="width:240px;height:10px;background:linear-gradient(90deg,{stops})"></div>'
            f'<span>{vmax:.3g}</span></div></div>')


def show_grid(items, cols=6, title=None, note=None, cap_width=HEAT_SIZE + 20):
    cells = "".join(
        f'<div style="text-align:center;font:11px sans-serif">{svg}'
        f'<div style="max-width:{cap_width}px;margin:auto">{cap}</div></div>'
        for svg, cap in items
    )
    head = f'<div style="font:bold 14px sans-serif;margin:12px 0 6px">{title}</div>' if title else ""
    emit(f'<div class="block">{head}<div style="display:grid;grid-template-columns:repeat({cols},auto);'
         f'gap:6px;justify-content:start">{cells}</div>{note or ""}</div>')


def piece_grid(mat, lim, cmap="RdBu_r"):
    return [(heat_svg(mat[offset + j], -lim, lim, cmap), f"{side} {lab}")
            for side, offset in (("own", 0), ("opponent", 6))
            for j, lab in enumerate(PIECE_LABELS)]


def board_report(name, pool, first=False):
    cls = "" if first else ' class="section"'
    emit(f'<h2{cls} style="font:bold 18px sans-serif;margin:24px 0 4px;border-bottom:1px solid #dddbd3">'
         f'Time pressure: {name}</h2>')
    fails = pool[~pool["maia_correct"]]
    log(f"{name}: {len(pool)} non-forced positions | failure rate {(~pool['maia_correct']).mean():.3f} | "
        f"phase mix {pool['phase'].value_counts(normalize=True).round(2).to_dict()}")

    if SHOW["pca"] or SHOW["occupancy"]:
        sub = pool.sample(min(N_BOARD, len(pool)), random_state=42)
        B = np.stack([board_vec(f) for f in tqdm(sub["fen"], desc=f"encoding {name}")])
        failed = ~sub["maia_correct"].to_numpy()

    if SHOW["pca"]:
        bpca = PCA(n_components=5, random_state=42).fit(B)
        scores = bpca.transform(B)
        for k in range(2):
            comp = bpca.components_[k].reshape(12, 64)
            lim = np.abs(comp).max()
            show_grid(
                piece_grid(comp, lim),
                title=f"PC{k + 1} loadings, {name} ({bpca.explained_variance_ratio_[k]:.1%} of variance), side to move at the bottom",
                note=colorbar_html(-lim, lim, label="red = a piece on this square raises the PC score, blue = lowers it"),
            )
            dec = pd.Series(failed).groupby(pd.qcut(scores[:, k], 10, labels=False, duplicates="drop")).mean()
            log(f"{name} PC{k + 1} failure rate by score decile: {dec.round(3).tolist()}")

    if SHOW["occupancy"]:
        diff = (B[failed].mean(0) - B[~failed].mean(0)).reshape(12, 64) * 100
        lim = np.abs(diff).max()
        show_grid(
            piece_grid(diff, lim),
            title=f"Where pieces stand when Maia-2 fails vs. succeeds ({name}, n={len(sub)})",
            note=colorbar_html(-lim, lim, label="percentage points"),
        )

    if SHOW["squares"]:
        fails_black = fails["side"].eq("black").to_numpy()

        def square_share(ucis, part):
            c = np.zeros(64)
            for u, is_black in zip(ucis, fails_black):
                m = chess.Move.from_uci(u)
                sq = m.from_square if part == "from" else m.to_square
                c[chess.square_mirror(sq) if is_black else sq] += 1
            return c / c.sum() * 100

        for part in ("from", "to"):
            human = square_share(fails["move"], part)
            maia = square_share(fails["maia_move"], part)
            top = max(human.max(), maia.max())
            d = human - maia
            dl = np.abs(d).max()
            show_grid(
                [
                    (heat_svg(human, 0, top, "Blues"), f"human '{part}' squares"),
                    (heat_svg(maia, 0, top, "Blues"), f"Maia-2 '{part}' squares"),
                    (heat_svg(d, -dl, dl, "RdBu_r"), "human minus Maia-2"),
                ],
                cols=3,
                title=f"Failed predictions, {name}: '{part}' squares (n={len(fails)}, mover at the bottom)",
                note=colorbar_html(0, top, "Blues", "share of moves in %")
                + colorbar_html(-dl, dl, label="red = humans, blue = Maia-2"),
            )

    if SHOW["examples"]:
        ex = fails.drop_duplicates("fen").nlargest(N_EXAMPLES, "maia_p_top1")
        items = []
        for _, r in ex.iterrows():
            b = chess.Board(r["fen"])
            hm, mm = chess.Move.from_uci(r["move"]), chess.Move.from_uci(r["maia_move"])
            svg = chess.svg.board(
                b,
                arrows=[chess.svg.Arrow(mm.from_square, mm.to_square, color=BLUE_A),
                        chess.svg.Arrow(hm.from_square, hm.to_square, color=ORANGE_A)],
                size=EXAMPLE_SIZE,
                coordinates=True,
                flipped=b.turn == chess.BLACK,
            )
            cap = (f"{r['side']} to move, {r['elo_self']} vs {r['elo_oppo']}, {r['clock_before']:.0f}s left<br>"
                   f"Maia-2 {r['maia_move']} ({r['maia_p_top1']:.0%}) | human {r['move']} ({r['maia_p_true']:.0%})<br>"
                   f"{r['opening']}")
            items.append((svg, cap))
        show_grid(
            items,
            cols=4,
            cap_width=EXAMPLE_SIZE,
            title=f"Most confident Maia-2 failures ({name})",
            note='<div style="font:12px sans-serif;color:#52514e;margin-top:6px">blue arrow = Maia-2 top prediction, orange arrow = move actually played</div>',
        )


def find_browser():
    candidates = ["/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"]
    found = next((c for c in candidates if Path(c).exists()), None)
    return found or next((shutil.which(n) for n in ("google-chrome", "chromium", "chromium-browser", "chrome") if shutil.which(n)), None)


def export_report(stem, title):
    css = ("@page{size:A4 landscape;margin:10mm}"
           "body{font-family:sans-serif;color:#0b0b0b;background:#ffffff}"
           ".section{break-before:page}"
           ".block{break-inside:avoid;page-break-inside:avoid;margin-bottom:8px}")
    html = (f'<!doctype html><html><head><meta charset="utf-8"><title>{title}</title><style>{css}</style></head>'
            f'<body><h1 style="font:bold 22px sans-serif">{title}</h1>{"".join(REPORT)}</body></html>')
    html_path = Path(f"{stem}.html").resolve()
    pdf_path = Path(f"{stem}.pdf").resolve()
    html_path.write_text(html, encoding="utf-8")
    browser = find_browser()
    if browser:
        subprocess.run(
            [browser, "--headless=new", "--disable-gpu", "--no-pdf-header-footer",
             f"--print-to-pdf={pdf_path}", html_path.as_uri()],
            capture_output=True,
            timeout=300,
        )

for i, (name, pool) in enumerate(GROUPS.items()):
    board_report(name, pool, first=i == 0)
export_report(REPORT_NAME, REPORT_TITLE)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

base = moves[~moves["forced"]]
later = base[base["phase"].isin(["middlegame", "endgame"])]
ROWS = {
    "Phase": {ph: base[base["phase"] == ph] for ph in ["opening", "middlegame", "endgame"]},
    "Time (mid+endgame)": {
        "> 1 min": later[later["clock_before"] > 60],
        "30 s - 1 min": later[(later["clock_before"] <= 60) & (later["clock_before"] > 30)],
        "< 30 s": later[later["clock_before"] <= 30],
    },
}


def to_squares(ucis, is_black):
    file = ucis.str[2].map(ord).to_numpy() - ord("a")
    rank = ucis.str[3].astype(int).to_numpy() - 1
    rank = np.where(is_black, 7 - rank, rank)
    return rank * 8 + file


def fail_square_diff(df):
    f = df[~df["maia_correct"]]
    is_black = f["side"].eq("black").to_numpy()
    human = np.bincount(to_squares(f["move"], is_black), minlength=64)
    maia = np.bincount(to_squares(f["maia_move"], is_black), minlength=64)
    return ((human - maia) / len(f) * 100).reshape(8, 8), len(f)


fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.subplots_adjust(hspace=0.35, wspace=0.15, right=0.88)
all_results = {row_name: [(name, *fail_square_diff(df), len(df)) for name, df in groups.items()]
               for row_name, groups in ROWS.items()}
lim = np.percentile(np.abs(np.concatenate([r[1].ravel() for res in all_results.values() for r in res])), 98)
for row_axes, (row_name, results) in zip(axes, all_results.items()):
    for ax, (name, d, n_fail, n_all) in zip(row_axes, results):
        im = ax.imshow(np.clip(d, -lim, lim), origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
        ax.set_xticks(range(8), list("abcdefgh"))
        ax.set_yticks(range(8), [str(i) for i in range(1, 9)])
        ax.set_xticks(np.arange(-0.5, 8), minor=True)
        ax.set_yticks(np.arange(-0.5, 8), minor=True)
        ax.grid(which="minor", color="white", linewidth=1)
        ax.grid(which="major", visible=False)
        ax.tick_params(which="minor", length=0)
        for r in range(8):
            for c in range(8):
                if abs(d[r, c]) >= 0.5 * lim:
                    ax.text(c, r, f"{d[r, c]:+.1f}", ha="center", va="center", fontsize=8,
                            color="white" if abs(d[r, c]) >= 0.8 * lim else "black")
        ax.set_title(f"{row_name}: {name}\nfailed n={n_fail:,} | failure rate {n_fail / n_all:.2f}", fontsize=10)

fig.colorbar(im, ax=axes, shrink=0.85, label="human minus Maia-2")
fig.suptitle("Failed predictions\n"
             "red = humans move, blue = Maia-2 predicts",
             fontsize=12)
fig.savefig("maia2_square_diff_failures.pdf", bbox_inches="tight")
fig.savefig("maia2_square_diff_failures.png", dpi=200, bbox_inches="tight")
plt.show()